# 面试题：Self-Attention 如何从零实现，mask 和缩放到底解决什么问题？

## 面试回答主线

Self-Attention 先把同一序列线性映射成 Q、K、V，再用 `QKᵀ / √d_head` 计算每个位置对其他位置的相关分数。softmax 把分数变成权重，权重与 V 的加权和就是上下文化表示；多头机制让不同子空间学习不同关系。除以 `√d_head` 是为了防止维度增大时点积方差变大、softmax 过早饱和。padding mask 防止模型读取补齐位置，causal mask 则禁止解码位置读取未来 token。实现时最常见错误不是矩阵乘法本身，而是 mask 的方向、广播维度和使用有限负数后的泄漏。

## 真实案例：客服风险工单分类

每条工单都含一个常见业务主题，但真正标签取决于“未到账/已经到账”“重复扣款/只有一次”等上下文。数据是脱敏教学样本，只用于观察注意力和 mask，不代表线上分类质量。

In [1]:
import math  # 导入平方根以实现 scaled dot-product attention。
import torch  # 导入 PyTorch 以手写多头注意力并执行真实反向传播。
from torch import nn  # 导入基础模块和可学习参数抽象。
import torch.nn.functional as F  # 导入稳定二分类损失函数。
torch.set_num_threads(1)  # 小张量教学实验固定单线程以减少调度开销。
tickets = [  # 构造主题相同但上下文状态相反的客服工单。
    (["[CLS]", "退款", "至今", "未", "到账", "[SEP]"], 1),  # 退款未到账需要升级处理。
    (["[CLS]", "退款", "已经", "正常", "到账", "[SEP]"], 0),  # 已到账工单无需升级。
    (["[CLS]", "订单", "出现", "重复", "扣款", "[SEP]"], 1),  # 重复扣款属于资金风险。
    (["[CLS]", "订单", "只有", "一次", "扣款", "[SEP]"], 0),  # 单次正常扣款不升级。
    (["[CLS]", "账号", "突然", "无法", "登录", "[SEP]"], 1),  # 无法登录需要账户排障。
    (["[CLS]", "账号", "已经", "恢复", "登录", "[SEP]"], 0),  # 登录已恢复不升级。
    (["[CLS]", "包裹", "超时", "仍未", "送达", "[SEP]"], 1),  # 超时未送达需要物流介入。
    (["[CLS]", "包裹", "今天", "正常", "送达", "[SEP]"], 0),  # 正常送达无需升级。
    (["[CLS]", "优惠券", "错误", "无法", "使用", "[SEP]"], 1),  # 无法使用优惠券需要处理。
    (["[CLS]", "优惠券", "已经", "成功", "使用", "[SEP]"], 0),  # 已成功使用不升级。
]  # 结束客服工单列表。
print("工单文本                              标签  决策")  # 输出真实案例输入表标题。
for tokens, label in tickets:  # 逐条展示 token、标签和业务含义。
    decision = "升级" if label == 1 else "不升级"  # 把二元标签转换为可读决策。
    print(f"{' '.join(tokens):<36} {label:>4}  {decision}")  # 输出一条完整工单记录。

工单文本                              标签  决策
[CLS] 退款 至今 未 到账 [SEP]                  1  升级
[CLS] 退款 已经 正常 到账 [SEP]                 0  不升级
[CLS] 订单 出现 重复 扣款 [SEP]                 1  升级
[CLS] 订单 只有 一次 扣款 [SEP]                 0  不升级
[CLS] 账号 突然 无法 登录 [SEP]                 1  升级
[CLS] 账号 已经 恢复 登录 [SEP]                 0  不升级
[CLS] 包裹 超时 仍未 送达 [SEP]                 1  升级
[CLS] 包裹 今天 正常 送达 [SEP]                 0  不升级
[CLS] 优惠券 错误 无法 使用 [SEP]                1  升级
[CLS] 优惠券 已经 成功 使用 [SEP]                0  不升级


## Baseline（基线）：只看到主题关键词

若规则只检查“退款、扣款、登录、送达、使用”等主题词，十条工单都会命中，却无法区分问题仍存在还是已经解决。这个基线故意保留真实规则系统常见的上下文缺陷。

In [2]:
topic_keywords = {"退款", "扣款", "登录", "送达", "使用"}  # 定义只能识别业务主题的朴素关键词表。
baseline_predictions = []  # 创建列表保存逐工单规则预测。
print("工单主题   关键词命中  预测  标签")  # 输出基线逐样本结果表标题。
for tokens, label in tickets:  # 在全部工单上运行相同关键词规则。
    matched = sorted(topic_keywords.intersection(tokens))  # 找出当前工单出现的主题词。
    prediction = int(bool(matched))  # 任何主题命中都被错误视为需要升级。
    baseline_predictions.append(prediction)  # 保存当前规则预测供统一评估。
    print(f"{tokens[1]:<8} {str(matched):<12} {prediction:>4} {label:>4}")  # 输出主题、命中词、预测和真值。
baseline_accuracy = sum(prediction == label for prediction, (_, label) in zip(baseline_predictions, tickets)) / len(tickets)  # 计算关键词规则准确率。
print(f"主题关键词基线准确率：{baseline_accuracy:.1%}")  # 输出后续注意力模型的同数据对照。

工单主题   关键词命中  预测  标签
退款       ['退款']          1    1
退款       ['退款']          1    0
订单       ['扣款']          1    1
订单       ['扣款']          1    0
账号       ['登录']          1    1
账号       ['登录']          1    0
包裹       ['送达']          1    1
包裹       ['送达']          1    0
优惠券      ['使用']          1    1
优惠券      ['使用']          1    0
主题关键词基线准确率：50.0%


## 核心实现一：手写多头 Q/K/V、缩放、广播 mask 与合并

输入形状为 `[batch, length, model_dim]`，拆头后为 `[batch, heads, length, head_dim]`。key padding mask 广播到所有 query 和 head；causal mask 是上三角矩阵。函数返回每个 head 的完整注意力矩阵，方便审计而不是只返回最终 shape。

In [3]:
class MultiHeadSelfAttention(nn.Module):  # 定义不依赖现成 Attention 层的多头自注意力。
    def __init__(self, model_dim, head_count):  # 初始化 Q/K/V 与输出投影矩阵。
        super().__init__()  # 注册基础模块状态以追踪参数。
        if model_dim % head_count != 0:  # 每个 head 必须获得相同整数维度。
            raise ValueError("model_dim 必须能被 head_count 整除")  # 对错误架构配置给出明确失败信息。
        self.model_dim = model_dim  # 保存总隐藏维度。
        self.head_count = head_count  # 保存并行注意力头数量。
        self.head_dim = model_dim // head_count  # 计算每个 head 的子空间维度。
        self.query_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.12)  # 创建查询投影参数。
        self.key_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.12)  # 创建键投影参数。
        self.value_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.12)  # 创建值投影参数。
        self.output_weight = nn.Parameter(torch.randn(model_dim, model_dim) * 0.12)  # 创建多头合并后的输出投影参数。
    def split_heads(self, tensor):  # 把最后一维拆成 head_count 与 head_dim。
        batch_size, sequence_length, _ = tensor.shape  # 读取批量大小和序列长度。
        reshaped = tensor.reshape(batch_size, sequence_length, self.head_count, self.head_dim)  # 把隐藏维度拆为头和头内维度。
        return reshaped.transpose(1, 2)  # 调整为批量、头、序列、头内维度顺序。
    def forward(self, hidden, valid_mask=None, causal=False):  # 执行完整多头 scaled dot-product attention。
        queries = self.split_heads(hidden @ self.query_weight)  # 投影并拆分查询张量。
        keys = self.split_heads(hidden @ self.key_weight)  # 投影并拆分键张量。
        values = self.split_heads(hidden @ self.value_weight)  # 投影并拆分值张量。
        scores = queries @ keys.transpose(-2, -1) / math.sqrt(self.head_dim)  # 计算缩放后的全位置点积分数。
        if valid_mask is not None:  # 有 padding 信息时屏蔽所有无效 key 位置。
            key_mask = valid_mask[:, None, None, :].to(torch.bool)  # 把二维有效位扩展到 batch、head、query、key。
            scores = scores.masked_fill(~key_mask, torch.finfo(scores.dtype).min)  # 用当前浮点类型最小值阻断无效 key。
        if causal:  # 解码注意力需要额外禁止读取未来 token。
            sequence_length = hidden.shape[1]  # 读取当前序列长度以创建方形因果矩阵。
            future_mask = torch.triu(torch.ones(sequence_length, sequence_length, dtype=torch.bool), diagonal=1)  # 创建严格上三角未来位置标记。
            scores = scores.masked_fill(future_mask[None, None, :, :], torch.finfo(scores.dtype).min)  # 向 batch 和 head 广播因果屏蔽。
        attention = torch.softmax(scores, dim=-1)  # 沿 key 维把分数归一化为概率。
        head_outputs = attention @ values  # 对每个 head 的值向量执行加权求和。
        merged = head_outputs.transpose(1, 2).reshape(hidden.shape[0], hidden.shape[1], self.model_dim)  # 把各头恢复为连续隐藏维度。
        output = merged @ self.output_weight  # 用输出投影混合多个 head 的信息。
        return output, attention, (queries, keys, values, scores)  # 返回上下文、权重和全部关键中间张量。
torch.manual_seed(17)  # 固定一次独立的机制演示初始化。
preview_attention = MultiHeadSelfAttention(8, 2)  # 创建八维两头注意力用于形状和缩放检查。
preview_hidden = torch.randn(2, 5, 8)  # 构造两个五长度序列的可复现输入。
preview_mask = torch.tensor([[1, 1, 1, 1, 1], [1, 1, 1, 0, 0]], dtype=torch.long)  # 第二条序列包含两个 padding 位置。
preview_output, preview_weights, preview_debug = preview_attention(preview_hidden, preview_mask)  # 运行完整前向传播并保留中间量。
preview_queries, preview_keys, preview_values, preview_scores = preview_debug  # 解包 Q/K/V 和缩放分数供展示。
print("hidden/Q/K/V/output 形状：", preview_hidden.shape, preview_queries.shape, preview_keys.shape, preview_values.shape, preview_output.shape)  # 输出每一步真实张量形状。
print("第二条样本 head0 的 attention：")  # 输出 padding mask 生效后的权重矩阵标题。
for row in preview_weights[1, 0]:  # 逐 query 位置展示第零个 head 的 key 权重。
    print([round(float(value), 4) for value in row])  # 输出包含被屏蔽列零概率的注意力行。

hidden/Q/K/V/output 形状： torch.Size([2, 5, 8]) torch.Size([2, 2, 5, 4]) torch.Size([2, 2, 5, 4]) torch.Size([2, 2, 5, 4]) torch.Size([2, 5, 8])
第二条样本 head0 的 attention：
[0.3616, 0.3078, 0.3306, 0.0, 0.0]
[0.3122, 0.3387, 0.349, 0.0, 0.0]
[0.3415, 0.3321, 0.3264, 0.0, 0.0]
[0.3188, 0.3392, 0.3421, 0.0, 0.0]
[0.3261, 0.3476, 0.3264, 0.0, 0.0]


## 核心实现二：把手写 Attention 放进可训练工单模型

模型拥有 token/position 参数表、自定义两头 Attention 和 `[CLS]` 分类头。没有使用现成 Transformer；训练循环真实执行 `forward → BCE → backward → 手动 SGD`，然后输出每个工单的风险概率与两头 `[CLS]` 权重。

In [4]:
vocabulary = sorted({token for tokens, _ in tickets for token in tokens})  # 收集教学工单出现的完整 token 集合。
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 创建确定性的 token id 映射。
input_ids = torch.tensor([[token_to_id[token] for token in tokens] for tokens, _ in tickets], dtype=torch.long)  # 把等长工单转换为批量 id 张量。
valid_mask = torch.ones_like(input_ids)  # 当前训练样本没有 padding，因此所有位置均有效。
labels = torch.tensor([label for _, label in tickets], dtype=torch.float32)  # 把升级标签转换为浮点监督张量。
class TicketAttentionClassifier(nn.Module):  # 定义使用自研多头注意力的工单分类网络。
    def __init__(self, vocabulary_size, model_dim, head_count, max_length):  # 初始化 embedding、位置表、Attention 和分类头。
        super().__init__()  # 注册基础模块状态。
        self.token_weight = nn.Parameter(torch.randn(vocabulary_size, model_dim) * 0.10)  # 创建可学习 token embedding 参数。
        self.position_weight = nn.Parameter(torch.randn(max_length, model_dim) * 0.05)  # 创建可学习绝对位置参数。
        self.attention = MultiHeadSelfAttention(model_dim, head_count)  # 组合前面手写的多头注意力模块。
        self.classifier_weight = nn.Parameter(torch.randn(model_dim) * 0.10)  # 创建 `[CLS]` 二分类权重。
        self.classifier_bias = nn.Parameter(torch.zeros(()))  # 创建标量二分类偏置。
    def forward(self, ids, mask, causal=False):  # 执行 embedding、self-attention 与 `[CLS]` 分类。
        sequence_length = ids.shape[1]  # 读取当前输入序列长度。
        hidden = self.token_weight[ids] + self.position_weight[:sequence_length].unsqueeze(0)  # 相加 token 与绝对位置表示。
        contextual, attention, debug = self.attention(hidden, mask, causal)  # 调用自定义 Attention 生成上下文表示。
        logits = contextual[:, 0, :] @ self.classifier_weight + self.classifier_bias  # 只用 `[CLS]` 位置完成风险分类。
        return logits, attention, contextual, debug  # 返回分类分数和全部可解释中间量。
torch.manual_seed(29)  # 固定训练模型的参数初始化。
ticket_model = TicketAttentionClassifier(len(vocabulary), 12, 2, 8)  # 创建十二维两头工单模型。
training_trace = []  # 保存代表轮次的损失和准确率。
for step in range(1201):  # 执行足够轮次形成清晰的风险概率间隔。
    logits, attention, contextual, debug = ticket_model(input_ids, valid_mask)  # 对全部工单执行真实前向传播。
    loss = F.binary_cross_entropy_with_logits(logits, labels)  # 计算稳定二分类损失。
    loss.backward()  # 对 embedding、Q/K/V、输出投影和分类头执行反向传播。
    with torch.no_grad():  # 参数更新不应进入下一轮计算图。
        for parameter in ticket_model.parameters():  # 逐个遍历所有可学习参数。
            parameter -= 0.10 * parameter.grad  # 使用固定学习率执行手动 SGD。
            parameter.grad.zero_()  # 清空梯度避免跨轮错误累积。
    if step in {0, 20, 100, 300, 600, 1200}:  # 记录能够说明收敛趋势的代表轮次。
        predictions = (torch.sigmoid(logits.detach()) >= 0.5).to(torch.float32)  # 把当前概率阈值化为类别。
        accuracy = float((predictions == labels).to(torch.float32).mean())  # 计算当前全批量准确率。
        training_trace.append((step, float(loss.detach()), accuracy))  # 保存轮次、损失和准确率。
print("轮次 | BCE loss | 准确率")  # 输出真实训练轨迹表标题。
for step, loss_value, accuracy in training_trace:  # 逐个展示关键训练状态。
    print(f"{step:>4} | {loss_value:>8.4f} | {accuracy:>6.1%}")  # 输出轮次、损失和同数据准确率。

轮次 | BCE loss | 准确率
   0 |   0.6929 |  60.0%
  20 |   0.6928 |  50.0%
 100 |   0.6925 |  70.0%
 300 |   0.6910 |  90.0%
 600 |   0.6686 | 100.0%
1200 |   0.0017 | 100.0%


## 结果表：概率与 `[CLS]` 两头注意力

每个 head 都有独立 Q/K/V 子空间。我们不把注意力权重当作完整因果解释，但它能检查模型是否只盯着 `[SEP]`、是否读取到“未/重复/恢复/成功”等决定状态的上下文。

In [5]:
with torch.no_grad():  # 评估阶段关闭梯度图以生成确定性输出。
    final_logits, final_attention, final_contextual, final_debug = ticket_model(input_ids, valid_mask)  # 对全量工单执行最终前向传播。
    final_probabilities = torch.sigmoid(final_logits)  # 把 logit 转换为升级概率。
    final_predictions = (final_probabilities >= 0.5).to(torch.float32)  # 用固定阈值生成最终类别。
model_accuracy = float((final_predictions == labels).to(torch.float32).mean())  # 计算注意力模型在教学集上的准确率。
print("工单                         标签  概率   head0[CLS]注意力          head1[CLS]注意力")  # 输出逐样本结果表标题。
for index, (tokens, label) in enumerate(tickets):  # 遍历每条工单及其真实标签。
    head_zero = [round(float(value), 2) for value in final_attention[index, 0, 0]]  # 提取第零头的 `[CLS]` 权重。
    head_one = [round(float(value), 2) for value in final_attention[index, 1, 0]]  # 提取第一头的 `[CLS]` 权重。
    print(f"{' '.join(tokens[1:-1]):<28} {label:>4} {float(final_probabilities[index]):>6.3f}  {str(head_zero):<28} {head_one}")  # 输出正文、标签、概率和两头权重。
print(f"准确率对照：主题关键词={baseline_accuracy:.1%}，手写 Self-Attention={model_accuracy:.1%}")  # 输出相同数据和指标下的方案对照。

工单                         标签  概率   head0[CLS]注意力          head1[CLS]注意力
退款 至今 未 到账                      1  0.999  [0.17, 0.17, 0.17, 0.17, 0.17, 0.17] [0.17, 0.17, 0.17, 0.17, 0.17, 0.17]
退款 已经 正常 到账                     0  0.000  [0.17, 0.17, 0.17, 0.17, 0.17, 0.17] [0.16, 0.17, 0.17, 0.17, 0.17, 0.17]
订单 出现 重复 扣款                     1  0.997  [0.17, 0.17, 0.17, 0.17, 0.17, 0.17] [0.17, 0.17, 0.17, 0.17, 0.17, 0.17]
订单 只有 一次 扣款                     0  0.003  [0.17, 0.17, 0.17, 0.17, 0.17, 0.17] [0.17, 0.17, 0.17, 0.17, 0.17, 0.17]
账号 突然 无法 登录                     1  0.999  [0.17, 0.17, 0.17, 0.17, 0.17, 0.17] [0.17, 0.17, 0.17, 0.17, 0.17, 0.17]
账号 已经 恢复 登录                     0  0.001  [0.17, 0.17, 0.17, 0.17, 0.17, 0.17] [0.16, 0.17, 0.17, 0.17, 0.17, 0.17]
包裹 超时 仍未 送达                     1  0.998  [0.17, 0.17, 0.17, 0.17, 0.17, 0.17] [0.17, 0.17, 0.17, 0.17, 0.17, 0.17]
包裹 今天 正常 送达                     0  0.003  [0.17, 0.17, 0.17, 0.17, 0.17, 0.17] [0.16, 0.17, 0.17, 0.17, 0.17, 0.17]

## 结果解读

关键词基线只识别主题，所以问题已解决的五条工单仍被误报。本次小数据中两头的 `[CLS]` 权重接近均匀，模型主要靠 V 投影与分类头汇聚不同状态 token，仍把正负工单拉开；这恰好说明“权重不尖锐”不等于模型没用上下文，也不能把 attention 当作因果解释。缩放发生在 softmax 之前；若遗漏 `√d_head`，头维增大时分数幅度和梯度都会改变，不能靠后处理 attention 权重补救。

## 失败案例：Decoder 忘记 causal mask 会偷看未来

构造两条前缀完全相同、只在最后一个未来 token 不同的隐藏序列。双向 Attention 在位置 2 的输出会随未来变化；打开 causal mask 后，位置 2 只能读取 `0..2`，两次输出应逐位一致。

In [6]:
torch.manual_seed(101)  # 固定未来泄漏反例的隐藏状态。
prefix_hidden = torch.randn(1, 6, 12)  # 构造一条六位置的解码隐藏序列。
changed_future_hidden = prefix_hidden.clone()  # 复制序列以保持所有前缀位置完全相同。
changed_future_hidden[:, 5, :] += 8.0  # 只显著修改位置五这个未来 token。
full_mask = torch.ones(1, 6, dtype=torch.long)  # 标记六个位置都不是 padding。
unmasked_original = ticket_model.attention(prefix_hidden, full_mask, causal=False)[0]  # 错误地允许所有位置双向读取。
unmasked_changed = ticket_model.attention(changed_future_hidden, full_mask, causal=False)[0]  # 在未来 token 改变后重复错误计算。
causal_original = ticket_model.attention(prefix_hidden, full_mask, causal=True)[0]  # 使用因果 mask 计算原序列。
causal_changed = ticket_model.attention(changed_future_hidden, full_mask, causal=True)[0]  # 使用因果 mask 计算未来已改变的序列。
unmasked_leak = float((unmasked_original[:, 2, :] - unmasked_changed[:, 2, :]).abs().max())  # 量化双向 Attention 对未来变化的敏感度。
causal_leak = float((causal_original[:, 2, :] - causal_changed[:, 2, :]).abs().max())  # 量化因果 Attention 在相同前缀上的差异。
causal_row = ticket_model.attention(prefix_hidden, full_mask, causal=True)[1][0, 0, 2]  # 取得位置二在第零头上的完整因果权重行。
print(f"未来 token 改变后，未加 causal mask 的位置2最大漂移：{unmasked_leak:.6f}")  # 输出可复现的未来信息泄漏。
print(f"未来 token 改变后，加 causal mask 的位置2最大漂移：{causal_leak:.6f}")  # 输出修复后的前缀一致性。
print("位置2的因果注意力行：", [round(float(value), 4) for value in causal_row])  # 展示所有未来列的权重确实为零。

未来 token 改变后，未加 causal mask 的位置2最大漂移：1.841642
未来 token 改变后，加 causal mask 的位置2最大漂移：0.000000
位置2的因果注意力行： [0.3368, 0.3742, 0.289, 0.0, 0.0, 0.0]


## 生产差距与落地清单

教学模型只有两头和十条样本，线上需要 fused kernel、混合精度、dropout、残差、归一化以及严格的 padding/causal/滑窗组合 mask。发布前应做全量与增量解码一致性、不同 padding 长度一致性、全 mask 行、极长序列和 fp16 最小值测试；监控 attention 熵、NaN、时延、显存与业务分组误报。注意力权重可用于诊断，但不能单独作为模型决策的因果解释。

## 最小回归测试

断言只保护张量合同、mask 和核心效果，完整证据来自训练轨迹、逐工单概率与未来泄漏对照。

In [7]:
assert preview_output.shape == preview_hidden.shape  # 验证多头合并后保持原始批量、长度和隐藏维度。
assert float(preview_weights[1, 0, :, 3:].abs().max()) == 0.0  # 验证 padding key 在所有 query 上得到零权重。
assert torch.allclose(preview_weights.sum(dim=-1), torch.ones_like(preview_weights.sum(dim=-1)))  # 验证每个 head 每行注意力概率归一化。
assert model_accuracy > baseline_accuracy  # 验证手写 Self-Attention 在同数据上优于主题关键词基线。
assert unmasked_leak > 1e-4  # 固化未加 causal mask 时未来 token 污染前缀的反例。
assert causal_leak < 1e-7  # 验证 causal mask 使相同前缀表示完全一致。
assert float(causal_row[3:].abs().max()) == 0.0  # 验证位置二对所有未来列的注意力严格为零。
print("最小回归测试通过：多头形状、概率归一化、分类效果和 causal 隔离均符合预期。")  # 输出完整顺序执行成功的明确结论。

最小回归测试通过：多头形状、概率归一化、分类效果和 causal 隔离均符合预期。
